## Test if my GPU supports P2P

In [1]:
import torch

def check_p2p_support():
    if not torch.cuda.is_available():
        print("CUDA is not available. No GPUs detected.")
        return

    num_gpus = torch.cuda.device_count()
    print(f"Found {num_gpus} GPU(s)")

    for i in range(num_gpus):
        for j in range(num_gpus):
            if i == j:
                continue  # Skip self-check

            # Check if P2P access is possible
            try:
                # Enable P2P access (temporarily)
                torch.cuda.set_device(i)  # Set current GPU
                can_access = torch.cuda.can_device_access_peer(i,j)
                print(f"GPU {i} can access GPU {j} via P2P: {'✅ Yes' if can_access else '❌ No'}")
            except RuntimeError as e:
                print(f"P2P between GPU {i} and GPU {j} not supported: {e}")

check_p2p_support()

Found 2 GPU(s)
GPU 0 can access GPU 1 via P2P: ❌ No
GPU 1 can access GPU 0 via P2P: ❌ No


## Pybind Import

In [2]:
import torch
import json
import requests
import time

from tensor_utils_pybind import get_ipc_handle_pybind, tensor_restore_from_handler_pybind,merge_tensors_and_export_ipc_handle

token_num = 4096

def get_dtype_size(dtype):
    """获取dtype的字节大小"""
    return torch.tensor([], dtype=dtype).element_size()

## warmup

In [3]:
token_num = 1024
tensorA = torch.randn( token_num, 2048, dtype=torch.float32).to("cuda")
tensorB = torch.randint(0, 60, (token_num, 4), dtype=torch.int32).to("cuda")
tensorC= torch.randn(token_num, 4, dtype=torch.bfloat16).to("cuda")

warmup_tensors=[tensorA, tensorB, tensorC]

warmup_handler = merge_tensors_and_export_ipc_handle(warmup_tensors, tensorA.device.index)
# completed preparing handler

max_dtype = max(warmup_tensors, key=lambda t: get_dtype_size(t.dtype)).dtype
max_dtype_size = get_dtype_size(max_dtype)
total_elements = 0
warmup_metadata = []
offset_bytes = 0
for tensor in warmup_tensors:
    # 计算当前张量需要的元素数（考虑对齐）
    tensor_bytes = tensor.numel() * get_dtype_size(tensor.dtype)
    elements_needed = (tensor_bytes + max_dtype_size - 1) // max_dtype_size
    
    # 记录元数据
    warmup_metadata.append({
        'dtype': str(tensor.dtype),
        'shape': tensor.shape,
        'device': tensor.device.index,
        'offset_bytes':offset_bytes
    })
    offset_bytes += tensor_bytes
    total_elements += elements_needed
# completed preparing metadata


url1 = "http://localhost:1177/merged_handler"

response_warmup = requests.post(url1, 
  data={
        'hidden_states_meta': json.dumps(warmup_metadata[0]),
        'topk_weights_meta': json.dumps(warmup_metadata[1]),
        'topk_ids_meta': json.dumps(warmup_metadata[2]),
},
                          
files={
        'handler': ('handler.bin', warmup_handler, 'application/octet-stream'),
})

print(response_warmup.json())
print("finished warmup")



{'message': 'ok'}
finished warmup


## handler preparation

In [4]:
token_num = 1024
hidden_states = torch.randn( token_num, 2048, dtype=torch.float32).to("cuda")
topk_ids = torch.randint(0, 60, (token_num, 4), dtype=torch.int32).to("cuda")
topk_weights = torch.randn(token_num, 4, dtype=torch.bfloat16).to("cuda")

start_handler_prepare = time.time()
tensors=[hidden_states,topk_weights,topk_ids]

handler = merge_tensors_and_export_ipc_handle(tensors,hidden_states[0].device.index)
end_handler_prepare = time.time()

print(f"It takes {(end_handler_prepare-start_handler_prepare)*1000}:.2f ms to prepare handler.")


It takes 0.5044937133789062:.2f ms to prepare handler.


## metadata preparation


In [5]:

start_metadata_prepare = time.time()

max_dtype = max(tensors, key=lambda t: get_dtype_size(t.dtype)).dtype
max_dtype_size = get_dtype_size(max_dtype)
print(f"max_dtype: {max_dtype}, max_dtype_size: {max_dtype_size} ")

# 2. 计算总元素数（考虑对齐）
total_elements = 0
metadata = []
offset_bytes = 0
for tensor in tensors:
    # 计算当前张量需要的元素数（考虑对齐）
    tensor_bytes = tensor.numel() * get_dtype_size(tensor.dtype)
    elements_needed = (tensor_bytes + max_dtype_size - 1) // max_dtype_size
    
    # 记录元数据
    metadata.append({
        'dtype': str(tensor.dtype),
        'shape': tensor.shape,
        'device': tensor.device.index,
        'offset_bytes':offset_bytes
    })
    offset_bytes += tensor_bytes
    total_elements += elements_needed
    
end_metadata_prepare = time.time()
print(f"It takes {(end_metadata_prepare-start_metadata_prepare)*1000}:.2f ms to prepare metadata.")

print(f"total_elements: {total_elements}")


max_dtype: torch.float32, max_dtype_size: 4 
It takes 1.2938976287841797:.2f ms to prepare metadata.
total_elements: 2103296


## Send requests

### Data check

In [6]:
print(f"tensors: {tensors}")
print(f"metadata: {metadata}")

tensors: [tensor([[ 0.5085, -0.6637,  0.8721,  ..., -0.1993,  1.2322, -0.2183],
        [ 0.2567, -0.1289, -2.6810,  ..., -0.4056, -1.6370, -1.2568],
        [-0.9615,  0.6709, -0.9246,  ..., -0.7930, -0.1826, -0.3114],
        ...,
        [ 1.4385, -0.5238, -1.1098,  ..., -0.8638, -0.4459, -1.0100],
        [ 0.7673,  2.0026,  0.0254,  ...,  0.3831,  0.0618,  0.4125],
        [-0.8063,  1.0232,  0.5609,  ..., -0.7465, -0.3194,  0.5995]],
       device='cuda:1'), tensor([[ 0.9414,  1.3281, -1.6094,  0.3496],
        [-0.7578,  2.7344,  0.0322,  0.1279],
        [ 1.4844,  0.7109,  0.4414,  0.4746],
        ...,
        [ 0.4395,  2.4375, -0.8164,  1.8750],
        [ 0.3340,  0.1934, -0.2637,  1.0938],
        [-0.6406, -1.5469,  0.1689, -0.3613]], device='cuda:1',
       dtype=torch.bfloat16), tensor([[18, 29, 40,  8],
        [19, 15, 57, 31],
        [ 9, 12, 54, 44],
        ...,
        [22, 14, 12, 37],
        [11, 27, 33,  8],
        [ 2, 42, 49, 52]], device='cuda:1', dtype=t

### Sending requests and in the server side restore multiple tensors

In [7]:
url1 = "http://localhost:1177/merged_handler"

#2101248
start_transmission=time.time()
response1 = requests.post(url1, 
  data={
        'hidden_states_meta': json.dumps(metadata[0]),
        'topk_weights_meta': json.dumps(metadata[1]),
        'topk_ids_meta': json.dumps(metadata[2]),
},

 
                          
files={
        'handler': ('handler.bin', handler, 'application/octet-stream'),
})

response_json= response1.json()
end_transmission=time.time()

print(f" transmission and restoration costs {(end_transmission-start_transmission)*1000:.2f} ms")


 transmission and restoration costs 9.99 ms


### Sending requests and in the server side restore 1 tensor

In [8]:
# md={'dtype': 'torch.float32',
#  'shape': [1024*2048+4096],
#  'offset': 0,
#  'elements': 1024*2048+4096,
#  'device': 1,
#  'offset_bytes': 0}

In [9]:
# import json
# import requests
# url1 = "http://localhost:1177/merged_single"

# #2101248

# response1 = requests.post(url1, 
#   data={
#         'hidden_states_meta': json.dumps(md),
#         # 'topk_ids_meta': json.dumps(metadata[2]),
# },
                          
# files={
#         'merged_handler': ('merged_handler.bin', merged_handler, 'application/octet-stream'),
# })
